# 01 — Exploratory Data Analysis
**Project:** Olist Brazilian E-Commerce — Customer Behavior & Cohort Analysis  
**Dataset:** [Kaggle: Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  

This notebook loads all nine Olist datasets, checks data quality, and produces summary statistics and initial visualizations that frame the rest of the analysis.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw, build_master, build_items_master
from src.utils import apply_style, PALETTE

apply_style()
print('Libraries loaded ✓')

## 1.1 Load All Datasets

In [ ]:
raw = load_raw()

for name, df in raw.items():
    print(f'{name:<25} → {df.shape[0]:>7,} rows  ×  {df.shape[1]:>2} cols')

## 1.2 Data Quality Check

In [ ]:
orders = raw['orders']

print('=== Orders — Null counts ===')
print(orders.isnull().sum())
print()
print('=== Order Status Distribution ===')
print(orders['order_status'].value_counts())

## 1.3 Build Master Table (Delivered Orders Only)

In [ ]:
master = build_master(raw)
print(f'Master table shape: {master.shape}')
print(f"Date range: {master['order_purchase_timestamp'].min().date()} → {master['order_purchase_timestamp'].max().date()}")
master.head(3)

## 1.4 Key Summary Statistics

In [ ]:
print(f"Total delivered orders : {len(master):,}")
print(f"Unique customers       : {master['customer_unique_id'].nunique():,}")
print(f"Total revenue          : R$ {master['payment_value'].sum():,.2f}")
print(f"Average order value    : R$ {master['payment_value'].mean():,.2f}")
print(f"Median order value     : R$ {master['payment_value'].median():,.2f}")
print(f"Late deliveries        : {master['is_late_delivery'].mean()*100:.1f}%")
print()
master[['payment_value', 'delivery_days']].describe().round(2)

## 1.5 Monthly Revenue Trend

In [ ]:
master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M')
monthly = master.groupby('order_month').agg(
    revenue=('payment_value', 'sum'),
    orders=('order_id', 'count')
).reset_index()
monthly['order_month_dt'] = monthly['order_month'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.fill_between(monthly['order_month_dt'], monthly['revenue']/1000, alpha=0.25, color=PALETTE['primary'])
ax1.plot(monthly['order_month_dt'], monthly['revenue']/1000, color=PALETTE['primary'], lw=2.5, label='Revenue (R$K)')
ax2.bar(monthly['order_month_dt'], monthly['orders'], width=20, alpha=0.4, color=PALETTE['secondary'], label='Orders')
ax1.set_ylabel('Revenue (R$ thousands)')
ax2.set_ylabel('Order Count')
ax1.set_title('Monthly Revenue & Order Volume', fontsize=14)
plt.tight_layout()
plt.show()

## 1.6 Geographic Distribution

In [ ]:
state_rev = master.groupby('customer_state').agg(
    orders=('order_id', 'count'),
    revenue=('payment_value', 'sum')
).sort_values('revenue', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Blues_r([i/len(state_rev) for i in range(len(state_rev))])
ax.barh(state_rev.index[::-1], state_rev['revenue'][::-1]/1000, color=colors[::-1])
ax.set_xlabel('Revenue (R$ thousands)')
ax.set_title('Top 10 States by Revenue', fontsize=13)
plt.tight_layout()
plt.show()

print(state_rev.to_string())

---
**Next:** [02_rfm_analysis.ipynb](02_rfm_analysis.ipynb)